# Gradio


In [1]:
!pip install gradio
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=e9680d3eee812b8a5d30bdd7c807baa8d1e8e47c8cd1e2ae91049c5191e08f28
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from lime.lime_tabular import LimeTabularExplainer
import gradio as gr
import numpy as np
import matplotlib.pyplot as plt


In [3]:

df = pd.read_csv("bank-full.csv", sep=';')

X = df.drop(columns=["y"])
y = df["y"]


In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
num_cols = ['age','balance','duration','day','pdays','previous','campaign']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age', 'balance', 'duration',
                                                   'day', 'pdays', 'previous',
                                                   'campaign']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('clf', RandomForestClassifier(random_state=42))])

In [6]:
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[7774  211]
 [ 629  429]]
              precision    recall  f1-score   support

          no       0.93      0.97      0.95      7985
         yes       0.67      0.41      0.51      1058

    accuracy                           0.91      9043
   macro avg       0.80      0.69      0.73      9043
weighted avg       0.90      0.91      0.90      9043



In [7]:
def predict(age, balance, duration, day, pdays, previous, campaign,
            job, marital, education, contact, month, poutcome,
            default, housing, loan):

    row = pd.DataFrame([locals()])
    proba = model.predict_proba(row)[0]

    return {"no": float(proba[0]), "yes": float(proba[1])}


In [8]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="age"),
        gr.Number(label="balance"),
        gr.Number(label="duration"),
        gr.Number(label="day"),
        gr.Number(label="pdays"),
        gr.Number(label="previous"),
        gr.Number(label="campaign"),
        gr.Dropdown(df["marital"].unique().tolist(), label="marital"),
        gr.Dropdown(df["education"].unique().tolist(), label="education"),
        gr.Dropdown(df["contact"].unique().tolist(), label="contact"),
        gr.Dropdown(df["month"].unique().tolist(), label="month"),
        gr.Dropdown(df["poutcome"].unique().tolist(), label="poutcome"),
        gr.Radio(["no","yes"], label="default"),
        gr.Radio(["no","yes"], label="housing"),
        gr.Radio(["no","yes"], label="loan"),
    ],
    outputs=gr.Label(label="Probabilidades"),
    title="Predicción de Depósito Bancario"
)

demo.launch(inline=True, share=False)


/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 16 arguments for function <function predict at 0x7bc2fec56c00>, received 15.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1056: UserWarning: Expected at least 16 arguments for function <function predict at 0x7bc2fec56c00>, received 15.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### podemos hacer cosas más complejas

In [9]:
prep = model.named_steps["prep"]
clf = model.named_steps["clf"]

X_train_enc = prep.transform(X_train)

ohe = prep.named_transformers_["cat"]
feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))

class_names = list(clf.classes_)

explainer = LimeTabularExplainer(
    training_data=np.asarray(X_train_enc),
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True
)

def predict_proba_encoded(X_enc):
    return clf.predict_proba(X_enc)

def lime_explain_instance(row_raw: pd.Series, num_features: int = 12):
    row_df = row_raw.to_frame().T
    row_enc = prep.transform(row_df)
    exp = explainer.explain_instance(
        data_row=np.asarray(row_enc)[0],
        predict_fn=predict_proba_encoded,
        num_features=num_features
    )
    return exp

In [10]:
def uniq(col):
    return sorted(df[col].dropna().unique().tolist())

def make_row(age, balance, duration, day, pdays, previous, campaign,
             job, marital, education, contact, month, poutcome,
             default, housing, loan):
    return pd.DataFrame([{
        "age": age, "balance": balance, "duration": duration, "day": day,
        "pdays": pdays, "previous": previous, "campaign": campaign,
        "job": job, "marital": marital, "education": education, "contact": contact,
        "month": month, "poutcome": poutcome,
        "default": default, "housing": housing, "loan": loan
    }])

def predict_and_explain(age, balance, duration, day, pdays, previous, campaign,
                        job, marital, education, contact, month, poutcome,
                        default, housing, loan, lime_features):
    row = make_row(age, balance, duration, day, pdays, previous, campaign,
                   job, marital, education, contact, month, poutcome,
                   default, housing, loan)

    proba = model.predict_proba(row)[0]
    pred = model.predict(row)[0]

    exp = lime_explain_instance(row.iloc[0], num_features=int(lime_features))

    classes = list(model.named_steps["clf"].classes_)
    label_idx = classes.index("yes") if "yes" in classes else 1

    fig = exp.as_pyplot_figure(label=label_idx)
    plt.tight_layout()

    weights = exp.as_list(label=label_idx)
    df_weights = pd.DataFrame(weights, columns=["feature", "weight"])

    return (
        {"no": float(proba[0]), "yes": float(proba[1])},
        pred,
        df_weights,
        fig
    )

def load_example(idx: int):
    row = X_test.iloc[int(idx)]
    return (
        float(row["age"]), float(row["balance"]), float(row["duration"]), float(row["day"]),
        float(row["pdays"]), float(row["previous"]), float(row["campaign"]),
        row["job"], row["marital"], row["education"], row["contact"],
        row["month"], row["poutcome"], row["default"], row["housing"], row["loan"]
    )

with gr.Blocks(title="Ejercicio del Banco") as demo:
    gr.Markdown("## Predicción de Depósito Bancario (Gradio Blocks + LIME)")
    gr.Markdown("Ntic 2025/2026")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Inputs")
            age = gr.Number(label="age", value=float(df["age"].median()))
            balance = gr.Number(label="balance", value=float(df["balance"].median()))
            duration = gr.Number(label="duration", value=float(df["duration"].median()))
            day = gr.Number(label="day", value=float(df["day"].median()))
            pdays = gr.Number(label="pdays", value=float(df["pdays"].median()))
            previous = gr.Number(label="previous", value=float(df["previous"].median()))
            campaign = gr.Number(label="campaign", value=float(df["campaign"].median()))

            job = gr.Dropdown(choices=uniq("job"), label="job", value=uniq("job")[0])
            marital = gr.Dropdown(choices=uniq("marital"), label="marital", value=uniq("marital")[0])
            education = gr.Dropdown(choices=uniq("education"), label="education", value=uniq("education")[0])
            contact = gr.Dropdown(choices=uniq("contact"), label="contact", value=uniq("contact")[0])
            month = gr.Dropdown(choices=uniq("month"), label="month", value=uniq("month")[0])
            poutcome = gr.Dropdown(choices=uniq("poutcome"), label="poutcome", value=uniq("poutcome")[0])

            default = gr.Radio(["no","yes"], label="default", value="no")
            housing = gr.Radio(["no","yes"], label="housing", value="no")
            loan = gr.Radio(["no","yes"], label="loan", value="no")

            lime_features = gr.Slider(5, 20, value=12, step=1, label="LIME: nº de features a mostrar")

            btn = gr.Button("Predecir + Explicar")

            gr.Markdown("### Cargar ejemplo (X_test)")
            idx = gr.Slider(0, len(X_test)-1, value=0, step=1, label="Índice de ejemplo")
            btn_load = gr.Button("Cargar valores del ejemplo")

        with gr.Column(scale=1):
            with gr.Tabs():
                with gr.Tab("Predicción"):
                    out_label = gr.Label(label="Probabilidades")
                    out_pred = gr.Textbox(label="Predicción (clase)", interactive=False)
                with gr.Tab("LIME (tabla)"):
                    out_weights = gr.Dataframe(headers=["feature", "weight"], label="Top features (LIME)")
                with gr.Tab("LIME (HTML)"):
                    out_plot = gr.Plot(label="LIME (plot)")

    inputs = [age, balance, duration, day, pdays, previous, campaign,
              job, marital, education, contact, month, poutcome,
              default, housing, loan, lime_features]

    btn.click(
        fn=predict_and_explain,
        inputs=inputs,
        outputs=[out_label, out_pred, out_weights, out_plot]
    )

    btn_load.click(
        fn=load_example,
        inputs=[idx],
        outputs=[age, balance, duration, day, pdays, previous, campaign,
                 job, marital, education, contact, month, poutcome,
                 default, housing, loan]
    )

demo.launch(inline=True, share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>